# inspect_official_training

Prints the OmniGen2 training path verbatim: docs, configs, launch script, the LoRA and loss blocks of `train.py`, the training dataset class, the data-config format, and the checkpoint converter.

Read-only. No GPU, no model download, no API. Shallow clone only.

Section 6 searches `train.py` for a validation loop and reports what it finds.

## 1. Clone (skip if present)

In [1]:
import os, subprocess

REPO = "/content/OmniGen2"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/VectorSpaceLab/OmniGen2.git", REPO],
                   check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
    print("cloned")
else:
    print("already present")

rev = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO,
                     capture_output=True, text=True).stdout.strip()
print("commit:", rev)

cloned
commit: 18e6f9d5271b517fcb32e999f10df943ae9b8f20


## 2. Print helpers

In [2]:
import re, ast

def show(path, first=None, last=None, title=None):
    """Print a file with line numbers. first/last limit the range."""
    p = os.path.join(REPO, path)
    print("=" * 100)
    print(title or path)
    print("=" * 100)
    if not os.path.exists(p):
        print("MISSING:", p)
        return
    lines = open(p, encoding="utf-8").read().splitlines()
    a = (first or 1) - 1
    b = last or len(lines)
    for i in range(a, min(b, len(lines))):
        print(f"{i+1:>5} | {lines[i]}")
    print()

def func_src(path, name, cls=None):
    """Line range of a named function, resolved by AST rather than by grep."""
    src = open(os.path.join(REPO, path), encoding="utf-8").read()
    found = []
    def walk(node, inside):
        for n in ast.iter_child_nodes(node):
            if isinstance(n, ast.ClassDef):
                walk(n, n.name)
            elif isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) and n.name == name \
                 and (cls is None or inside == cls):
                found.append((n.lineno, n.end_lineno))
    walk(ast.parse(src), None)
    return found[0] if found else None

def grep(path, pattern, context=0):
    p = os.path.join(REPO, path)
    lines = open(p, encoding="utf-8").read().splitlines()
    rx = re.compile(pattern)
    hits = [i for i, l in enumerate(lines) if rx.search(l)]
    if not hits:
        print(f"  (no match for {pattern!r} in {path})")
        return []
    shown = set()
    for i in hits:
        for j in range(max(0, i - context), min(len(lines), i + context + 1)):
            if j not in shown:
                print(f"{j+1:>5} | {lines[j]}")
                shown.add(j)
    return hits

print("helpers ready")

helpers ready


## 3. docs/FINETUNE.md

In [3]:
show("docs/FINETUNE.md")

docs/FINETUNE.md
    1 | ## 🎨 Fine-Tuning OmniGen2
    2 | 
    3 | You can fine-tune OmniGen2 to customize its capabilities, enhance its performance on specific tasks, or address potential limitations.
    4 | 
    5 | We provide a training script that supports multi-GPU and multi-node distributed training using **PyTorch FSDP (Fully Sharded Data Parallel)**. Both full-parameter fine-tuning and **LoRA (Low-Rank Adaptation)** are supported out of the box.
    6 | 
    7 | ### 1. Preparation
    8 | 
    9 | Before launching the training, you need to prepare the following configuration files.
   10 | 
   11 | #### Step 1: Set Up the Training Configuration
   12 | 
   13 | This is a YAML file that specifies crucial parameters for your training job, including the model architecture, optimizer, dataset paths, and validation settings.
   14 | 
   15 | We provide two templates to get you started:
   16 | *   **Full-Parameter Fine-Tuning:** `options/ft.yml`
   17 | *   **LoRA Fine-Tuning:** `

## 4. Configs — full fine-tune vs LoRA

Both printed in full, then differenced. Everything outside the `lora_*` keys is shared, including `learning_rate`.

In [4]:
show("options/ft.yml")
show("options/ft_lora.yml")

import difflib
a = open(os.path.join(REPO, "options/ft.yml"), encoding="utf-8").read().splitlines()
b = open(os.path.join(REPO, "options/ft_lora.yml"), encoding="utf-8").read().splitlines()
print("=" * 100)
print("diff: ft.yml -> ft_lora.yml")
print("=" * 100)
for l in difflib.unified_diff(a, b, fromfile="ft.yml", tofile="ft_lora.yml", lineterm="", n=1):
    print(l)

options/ft.yml
    1 | name: ft
    2 | 
    3 | seed: 2233
    4 | device_specific_seed: true
    5 | workder_specific_seed: true
    6 | 
    7 | data:
    8 |   data_path: data_configs/train/example/mix.yml
    9 |   use_chat_template: true
   10 |   maximum_text_tokens: 888
   11 |   prompt_dropout_prob: !!float 0.0001
   12 |   ref_img_dropout_prob: !!float 0.5
   13 |   max_output_pixels: 1048576 # 1024 * 1024
   14 |   max_input_pixels: [1048576, 1048576, 589824, 262144] # [1024 * 1024, 1024 * 1024, 768 * 768, 512 * 512]
   15 |   max_side_length: 2048
   16 |   
   17 | model:
   18 |   pretrained_vae_model_name_or_path: black-forest-labs/FLUX.1-dev
   19 |   pretrained_text_encoder_model_name_or_path: Qwen/Qwen2.5-VL-3B-Instruct
   20 |   pretrained_model_path: OmniGen2/OmniGen2
   21 |   
   22 |   arch_opt:
   23 |     patch_size: 2
   24 |     in_channels: 16
   25 |     hidden_size: 2520
   26 |     num_layers: 32
   27 |     num_refiner_layers: 2
   28 |     num_attention

## 5. Launch scripts

`num_processes = WORLD_SIZE * 8` — the shipped script assumes 8 GPUs per node.

In [5]:
show("scripts/train/ft_lora.sh")
show("scripts/train/ft.sh")

scripts/train/ft_lora.sh
    1 | # !/bin/bash
    2 | SHELL_FOLDER=$(cd "$(dirname "$0")";pwd)
    3 | cd $(dirname $SHELL_FOLDER)
    4 | cd ../
    5 | 
    6 | source "$(dirname $(which conda))/../etc/profile.d/conda.sh"
    7 | conda activate py3.11+pytorch2.6+cu124
    8 | 
    9 | debug=false
   10 | RANK=0
   11 | MASTER_ADDR=1
   12 | MASTER_PORT=29500
   13 | WORLD_SIZE=1
   14 | 
   15 | # 处理命名参数
   16 | while [[ $# -gt 0 ]]; do
   17 |     case "$1" in
   18 |         --rank=*)
   19 |             RANK="${1#*=}"
   20 |             shift
   21 |             ;;
   22 |         --master_addr=*)
   23 |             MASTER_ADDR="${1#*=}"
   24 |             shift
   25 |             ;;
   26 |         --master_port=*)
   27 |             MASTER_PORT="${1#*=}"
   28 |             shift
   29 |             ;;
   30 |         --world_size=*)
   31 |             WORLD_SIZE="${1#*=}"
   32 |             shift
   33 |             ;;
   34 |         *)
   35 |             echo "未知参数: $

## 6. Validation search in train.py

Searches for a validation loop, a val loss, a val dataloader, and any read of the `val:` config block.

In [6]:
print("train.py lines:", len(open(os.path.join(REPO, "train.py"), encoding="utf-8").read().splitlines()))
print()

for pat in [r"def validat", r"validation_steps", r"val_loss", r"val_dataloader",
            r"valid_dataset", r"log_validation", r"eval_", r"torch\.no_grad"]:
    print(f"--- search: {pat}")
    grep("train.py", pat)
    print()

print("--- every read of args.val ---")
grep("train.py", r"args\.val", context=2)

print()
print("--- keys present in the val: block of ft_lora.yml ---")
from omegaconf import OmegaConf
cfg = OmegaConf.load(os.path.join(REPO, "options/ft_lora.yml"))
for k, v in cfg.val.items():
    used = bool(grep_silent := re.search(rf"args\.val\.{k}\b|'{k}' in args\.val",
                open(os.path.join(REPO, "train.py"), encoding="utf-8").read()))
    print(f"  val.{k} = {v}   ->  {'READ by train.py' if used else 'NEVER READ'}")

train.py lines: 721

--- search: def validat
  (no match for 'def validat' in train.py)

--- search: validation_steps
  (no match for 'validation_steps' in train.py)

--- search: val_loss
  (no match for 'val_loss' in train.py)

--- search: val_dataloader
  (no match for 'val_dataloader' in train.py)

--- search: valid_dataset
  (no match for 'valid_dataset' in train.py)

--- search: log_validation
  (no match for 'log_validation' in train.py)

--- search: eval_
  (no match for 'eval_' in train.py)

--- search: torch\.no_grad
  145 |     with torch.no_grad():
  514 |                 with torch.no_grad():
  521 |                 @torch.no_grad()
  644 |                         with torch.no_grad():

--- every read of args.val ---
  640 |                     
  641 |                 if accelerator.is_main_process:
  642 |                     if 'train_visualization_steps' in args.val and (global_step - 1) % args.val.train_visualization_steps == 0:
  643 |                         num_samp

## 7. train.py — LoRA setup

In [7]:
hits = grep("train.py", r"lora_ft = args\.train\.get")
if hits:
    s = hits[0] + 1
    show("train.py", first=s - 2, last=s + 20, title="train.py :: LoRA adapter construction")

  258 |     args.train.lora_ft = args.train.get('lora_ft', False)
train.py :: LoRA adapter construction
  256 |     text_encoder = text_encoder.to(accelerator.device, dtype=weight_dtype)
  257 |     
  258 |     args.train.lora_ft = args.train.get('lora_ft', False)
  259 |     if args.train.lora_ft:
  260 |         model.requires_grad_(False)
  261 | 
  262 |         target_modules = ["to_k", "to_q", "to_v", "to_out.0"]
  263 | 
  264 |         # now we will add new LoRA weights the transformer layers
  265 |         lora_config = LoraConfig(
  266 |             r=args.train.lora_rank,
  267 |             lora_alpha=args.train.lora_rank,
  268 |             lora_dropout=args.train.lora_dropout,
  269 |             init_lora_weights="gaussian",
  270 |             target_modules=target_modules,
  271 |         )
  272 |         model.add_adapter(lora_config)
  273 | 
  274 |     if args.train.gradient_checkpointing:
  275 |         model.enable_gradient_checkpointing()
  276 | 
  277 | 

## 8. train.py — loss, optimizer step, checkpointing

In [8]:
hits = grep("train.py", r"loss_dict = transport\.training_losses")
if hits:
    s = hits[0] + 1
    show("train.py", first=s - 6, last=s + 30, title="train.py :: loss computation")

hits = grep("train.py", r"checkpointing_steps")
if hits:
    s = hits[0] + 1
    show("train.py", first=s - 4, last=s + 30, title="train.py :: checkpoint save")

  557 |                 loss_dict = transport.training_losses(
train.py :: loss computation
  551 |                 local_num_tokens_in_batch = 0
  552 | 
  553 |                 for i, latent in enumerate(output_latents):
  554 |                     local_num_tokens_in_batch += latent.numel()
  555 |                 
  556 |                 num_tokens_in_batch = accelerator.gather(torch.tensor(local_num_tokens_in_batch, device=accelerator.device)).sum().item()
  557 |                 loss_dict = transport.training_losses(
  558 |                     model,
  559 |                     output_latents,
  560 |                     model_kwargs,
  561 |                     process_index=AcceleratorState().process_index,
  562 |                     num_processes=AcceleratorState().num_processes,
  563 |                     reduction='sum'
  564 |                 )
  565 |                 loss = loss_dict["loss"].sum()
  566 |                 loss = (loss * accelerator.gradient_state.num_ste

## 9. Training dataset

Field names, `task_type` handling, prompt dropout and reference dropout.

In [9]:
show("omnigen2/dataset/omnigen2_train_dataset.py")

omnigen2/dataset/omnigen2_train_dataset.py
    1 | from typing import Optional, Union, List
    2 | 
    3 | import os
    4 | import random
    5 | import yaml
    6 | import glob
    7 | from PIL import Image
    8 | 
    9 | import torch
   10 | from torchvision import transforms
   11 | 
   12 | from datasets import load_dataset, concatenate_datasets
   13 | 
   14 | from ..pipelines.omnigen2.pipeline_omnigen2 import OmniGen2ImageProcessor
   15 | 
   16 | class OmniGen2TrainDataset(torch.utils.data.Dataset):
   17 |     SYSTEM_PROMPT = "You are a helpful assistant that generates high-quality images based on user instructions."
   18 |     SYSTEM_PROMPT_DROP = "You are a helpful assistant that generates images."
   19 | 
   20 |     def __init__(
   21 |         self,
   22 |         config_path: str,
   23 |         tokenizer,
   24 |         use_chat_template: bool,
   25 |         max_input_pixels: Optional[Union[int, List[int]]] = None,
   26 |         max_output_pixels: Option

## 10. Data-config format

In [10]:
import glob
for p in sorted(glob.glob(os.path.join(REPO, "data_configs", "**", "*.yml"), recursive=True)):
    show(os.path.relpath(p, REPO))

for p in sorted(glob.glob(os.path.join(REPO, "data_configs", "**", "*.jsonl"), recursive=True))[:3]:
    print("=" * 100)
    print(os.path.relpath(p, REPO), "(first 3 lines)")
    print("=" * 100)
    import json
    for i, l in enumerate(open(p, encoding="utf-8")):
        if i >= 3:
            break
        print(json.dumps(json.loads(l), indent=2)[:800])
    print()

data_configs/train/example/edit/edit.yml
    1 | data:
    2 |   - 
    3 |     path: "data_configs/example/edit/jsonls/0.jsonl"
    4 |     type: "edit"
    5 |     ratio: !!float 1
    6 |   - 
    7 |     path: "data_configs/example/edit/jsonls/1.jsonl"
    8 |     type: "edit"
    9 |     ratio: !!float 1

data_configs/train/example/ic/ic.yml
    1 | data:
    2 |   - 
    3 |     path: "data_configs/example/ic/jsonls/0.jsonl"
    4 |     type: "ic"
    5 |     ratio: !!float 1
    6 |   - 
    7 |     path: "data_configs/example/ic/jsonls/1.jsonl"
    8 |     type: "ic"
    9 |     ratio: !!float 1

data_configs/train/example/mix.yml
    1 | data:
    2 |   - 
    3 |     path: 'data_configs/example/t2i/t2i.yml'
    4 |     type: 't2i'
    5 |     ratio: !!float 0.33
    6 |   -
    7 |     path: 'data_configs/example/edit/edit.yml'
    8 |     type: 'edit'
    9 |     ratio: !!float 0.33
   10 |   - 
   11 |     path: 'data_configs/example/ic/ic.yml'
   12 |     type: 'ic'
   13 

## 11. Checkpoint converter

In [11]:
show("convert_ckpt_to_hf_format.py")

convert_ckpt_to_hf_format.py
    1 | import dotenv
    2 | 
    3 | dotenv.load_dotenv(override=True)
    4 | 
    5 | import argparse
    6 | 
    7 | from omegaconf import OmegaConf
    8 | 
    9 | import torch
   10 | from accelerate import init_empty_weights
   11 | 
   12 | from peft import LoraConfig
   13 | from peft.utils import get_peft_model_state_dict
   14 | 
   15 | from omnigen2.models.transformers.transformer_omnigen2 import OmniGen2Transformer2DModel
   16 | from omnigen2.pipelines.omnigen2.pipeline_omnigen2 import OmniGen2Pipeline
   17 | 
   18 | 
   19 | def main(args):
   20 |     config_path = args.config_path
   21 |     model_path = args.model_path
   22 | 
   23 |     conf = OmegaConf.load(config_path)
   24 |     arch_opt = conf.model.arch_opt
   25 | 
   26 |     arch_opt = OmegaConf.to_object(arch_opt)
   27 |     # Convert lists to tuples in conf.model.arch_opt
   28 |     for key, value in arch_opt.items():
   29 |         if isinstance(value, list):
   30

## 12. Defaults that the thesis run would inherit

Values read straight from `ft_lora.yml`, alongside the thesis requirement for each.

In [12]:
from omegaconf import OmegaConf
c = OmegaConf.load(os.path.join(REPO, "options/ft_lora.yml"))

rows = [
    ("data.max_output_pixels",      c.data.max_output_pixels,      "512^2 = 262144 for this thesis"),
    ("data.max_input_pixels",       c.data.max_input_pixels,       "scale to 512 to match inference"),
    ("data.max_side_length",        c.data.max_side_length,        "512"),
    ("data.maximum_text_tokens",    c.data.maximum_text_tokens,    "inference uses 1024; prompts ~230"),
    ("data.ref_img_dropout_prob",   c.data.ref_img_dropout_prob,   "keep: gated on prompt dropout, effective rate 5e-5"),
    ("data.prompt_dropout_prob",    c.data.prompt_dropout_prob,    "keep"),
    ("train.global_batch_size",     c.train.global_batch_size,     "keep; 1 GPU needs grad_accum 8"),
    ("train.batch_size",            c.train.batch_size,            "per GPU"),
    ("train.gradient_accumulation_steps", c.train.gradient_accumulation_steps, "8 on a single GPU"),
    ("train.max_train_steps",       c.train.max_train_steps,       "keep"),
    ("train.learning_rate",         c.train.learning_rate,         "keep, given the full schedule"),
    ("train.warmup_t",              c.train.warmup_t,              "keep"),
    ("train.lora_rank",             c.train.lora_rank,             "thesis sweeps 4 / 8 / 16"),
    ("train.lora_dropout",          c.train.lora_dropout,          "keep"),
    ("train.gradient_checkpointing", c.train.gradient_checkpointing, "keep"),
    ("train.mixed_precision",       c.train.mixed_precision,       "keep"),
    ("val.validation_steps",        c.val.validation_steps,        "dead key, see section 6"),
    ("logger.checkpointing_steps",  c.logger.checkpointing_steps,  "500 to land 500 / 1500 / final"),
    ("logger.checkpoints_total_limit", c.logger.checkpoints_total_limit, "cap: LoRA saves the full model"),
]

w = max(len(r[0]) for r in rows)
print(f"{'key'.ljust(w)}  {'shipped default'.ljust(28)}  thesis requirement")
print("-" * (w + 60))
for k, v, note in rows:
    print(f"{k.ljust(w)}  {str(v).ljust(28)}  {note}")

print()
print("LoRA target modules (train.py and convert_ckpt_to_hf_format.py must agree):")
for f in ["train.py", "convert_ckpt_to_hf_format.py"]:
    grep(f, r"target_modules = ")

key                                shipped default               thesis requirement
---------------------------------------------------------------------------------------------
data.max_output_pixels             1048576                       512^2 = 262144 for this thesis
data.max_input_pixels              [1048576, 1048576, 589824, 262144]  scale to 512 to match inference
data.max_side_length               2048                          512
data.maximum_text_tokens           888                           inference uses 1024; prompts ~230
data.ref_img_dropout_prob          0.5                           keep: gated on prompt dropout, effective rate 5e-5
data.prompt_dropout_prob           0.0001                        keep
train.global_batch_size            16                            keep; 1 GPU needs grad_accum 8
train.batch_size                   2                             per GPU
train.gradient_accumulation_steps  1                             8 on a single GPU
train.max_train_s